<a href="https://colab.research.google.com/github/yudithvega-art/bhm-mrsa-indonesia/blob/main/Exposure_dose_full_rev_2_12092026.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

# Full exposure-dose Monte Carlo (10,000 runs)
Sources: Milk, Water, Bioaerosol, Dust.
Concentrations from BHM posteriors; finalized exposure-factor distributions.
Milk & water = CFU/event ; bioaerosol & dust = CFU/day.

**Outputs:** dose summary table + per-farm ridgeline figures (conditional filled + marginal dashed) + OFAT sensitivity.

**Run:** install → upload `bhm_summary.csv`, `bhm_by_farm.csv`, `bhm_prior_sensitivity.csv` → Run all.

In [ ]:
!pip -q install numpy pandas matplotlib scipy

In [ ]:
import os
need=['bhm_summary.csv','bhm_by_farm.csv','bhm_prior_sensitivity.csv']
if not all(os.path.exists(f) for f in need):
    try:
        from google.colab import files
        files.upload()
    except Exception as e:
        print('Upload the 3 CSVs', e)

Saving bhm_summary.csv to bhm_summary.csv
Saving bhm_prior_sensitivity.csv to bhm_prior_sensitivity.csv
Saving bhm_by_farm.csv to bhm_by_farm.csv


In [ ]:
# ============================================================================
#  Full exposure-dose Monte Carlo (10,000 runs) — presumptive MRSA
#  Sources: Milk, Water (POOLED), Bioaerosol, Dust.
#  Concentration posteriors from the BHM summary/per-farm CSVs; exposure factors
#  use the finalized distributions. Milk & water = CFU/event ; bioaerosol & dust = CFU/day.
#
#  Deliverables:
#    - dose summary table (conditional + marginal, per source)
#    - ridgeline per-farm figures (conditional + marginal on the same row), tidy spacing
#    - OFAT sensitivity figure (per source)
#
#  COLAB:  !pip -q install numpy pandas matplotlib scipy
#          upload  bhm_summary.csv , bhm_by_farm.csv , bhm_prior_sensitivity.csv
# ============================================================================
import numpy as np, pandas as pd
import matplotlib; # matplotlib.use("Agg")   # inline on Colab
import matplotlib.pyplot as plt
from matplotlib.lines import Line2D
from scipy.stats import gaussian_kde

rng = np.random.default_rng(42)
N   = 10000                                    # Monte-Carlo runs

summary = pd.read_csv("bhm_summary.csv").set_index("matrix")
byfarm  = pd.read_csv("bhm_by_farm.csv")
psens   = pd.read_csv("bhm_prior_sensitivity.csv")
OCC = psens[psens.prior_shift_log10 == 0].set_index("matrix")["occurrence"].to_dict()

SOURCES = ["Milk", "Water", "Bioaerosol", "Dust"]
UNIT  = {"Milk":"CFU/mL","Water":"CFU/100 mL","Bioaerosol":"CFU/m3","Dust":"CFU/cm2"}
DUNIT = {"Milk":"CFU/event","Water":"CFU/event","Bioaerosol":"CFU/day","Dust":"CFU/day"}
BASIS = {"Milk":"event","Water":"event","Bioaerosol":"day","Dust":"day"}
MC    = {"Milk":"#4A72A6","Water":"#2F8F8B","Bioaerosol":"#E07A5F","Dust":"#7B5EA7"}

# ---------- concentration posterior (lognormal from median + 95% CrI) ----------
def conc_summary(m, n=N):
    med, lo, hi = summary.loc[m, ["grand_mean","gm_lo","gm_hi"]]
    mu = np.log10(med); s = (np.log10(hi)-np.log10(lo))/(2*1.959964)
    return 10.0**rng.normal(mu, s, n)
def conc_farm(row, n=N):
    med, lo, hi = row["cond"], row["cond_lo"], row["cond_hi"]
    mu = np.log10(med); s = (np.log10(hi)-np.log10(lo))/(2*1.959964)
    return 10.0**rng.normal(mu, s, n)

# ---------- exposure-factor samplers (finalized distributions) ----------
def tri(a,c,b,n=N): return rng.triangular(a,c,b,n)
def f_milk(n=N):   # C[CFU/mL] * (h*A) * TE_hand-mouth * f_swallow  -> CFU/event
    return (tri(0.0038,0.0046,0.0078,n)*tri(3.56,68.7,133.75,n))*rng.beta(4.02,7.85,n)*rng.uniform(0.5,1,n)
def f_water(n=N):  # C[CFU/100mL] -> /100 * V_water(mL)  -> CFU/event
    return tri(0.5,1.0,5.0,n)/100.0
def f_bio(n=N):    # C[CFU/m3] * IR * t  -> CFU/day
    return tri(1.5,1.6,1.7,n)*tri(4.32,5.58,6.84,n)
def f_dust(n=N):   # C[CFU/cm2] * A * TE_sh * TE_hm * f_sw * N_events -> CFU/day
    return tri(3.56,68.7,133.75,n)*tri(0.01,0.208,0.406,n)*tri(0.03,0.10,0.15,n)*rng.uniform(0.5,1,n)*(tri(2,2.9,8,n)*tri(4.32,5.58,6.84,n))
FAC = {"Milk":f_milk,"Water":f_water,"Bioaerosol":f_bio,"Dust":f_dust}

# ---------- dose (matrix level): conditional and marginal ----------
def dose_matrix(m, n=N):
    Dc = conc_summary(m, n)*FAC[m](n)
    present = rng.random(n) < OCC[m]
    return Dc, np.where(present, Dc, 0.0)

def _q(x): return np.median(x), np.percentile(x,2.5), np.percentile(x,97.5), np.mean(x)

def dose_table():
    print("="*84)
    print(f"{'Source':11}{'dose unit':11}{'occ':6}{'conditional  median [95% CrI]':34}{'marginal mean'}")
    rows=[]
    for m in SOURCES:
        dc,dm = dose_matrix(m); c=_q(dc); mm=_q(dm)[3]
        print(f"{m:11}{DUNIT[m]:11}{OCC[m]:.2f}  {f'{c[0]:.3g} [{c[1]:.3g}, {c[2]:.4g}]':34}{mm:.4g}")
        rows.append(dict(source=m,dose_unit=DUNIT[m],occurrence=round(OCC[m],3),
                         cond_median=c[0],cond_lo=c[1],cond_hi=c[2],marg_mean=mm))
    pd.DataFrame(rows).to_csv("exposure_dose_summary.csv",index=False)
    print("Milk/water = CFU/event ; bioaerosol/dust = CFU/day. Presumptive MRSA. Saved: exposure_dose_summary.csv")

# ============================================================================
#  RIDGELINE per-farm (conditional filled + marginal dashed), tidy spacing
# ============================================================================
def ridgeline(source, fname=None):
    fname = fname or f"ridge_{source.lower()}.png"
    d = byfarm[byfarm.matrix==source].sort_values("farm").reset_index(drop=True)
    farms = d["farm"].tolist(); c = MC[source]
    Dc, allpos, occ_f = {}, [], {}
    for _,row in d.iterrows():
        cd = conc_farm(row)*FAC[source](N) if source!="Water" else (conc_farm(row)/100.0)*FAC["Water"](N)
        Dc[row["farm"]]=cd; occ_f[row["farm"]]=float(row["occ"])
        allpos.append(np.log10(cd[cd>0]))
    allv=np.concatenate(allpos); lo,hi=np.percentile(allv,[0.5,99.5]); xs=np.linspace(lo-0.5,hi+0.5,400)
    n=len(farms); SP=2.4; H=1.35                      # spacing / curve height (tidy, low overlap)
    fig,ax=plt.subplots(figsize=(9.4,0.95*n+1.8))
    for i,f in enumerate(farms[::-1]):
        y0=i*SP; cd=Dc[f]; occ=occ_f[f]
        lv=np.log10(np.clip(cd[cd>0],1e-8,None)); dens=gaussian_kde(lv)(xs); dens=dens/dens.max()*H
        ax.fill_between(xs,y0,y0+dens,color=c,alpha=0.55,lw=0,zorder=n-i)          # conditional
        ax.plot(xs,y0+dens,color=c,lw=1.3,zorder=n-i)
        ax.plot(xs,y0+dens*occ,color="#333",lw=1.1,ls="--",zorder=n-i+0.3)         # marginal (area x occ)
        ax.plot([np.log10(np.median(cd))]*2,[y0,y0+H*0.9],color="#111",lw=2.0,zorder=n-i+0.5)  # median
        ax.text(xs[-1]+0.05,y0+0.05,f"occ {occ:.2f}",fontsize=7.5,color="#666",va="bottom")
        ax.text(xs[0]-0.15,y0+0.05,f,ha="right",va="bottom",fontsize=10,fontweight="bold")
    ticks=list(range(int(np.floor(xs[0])),int(np.ceil(xs[-1]))+1))
    ax.set_xticks(ticks); ax.set_xticklabels([f"$10^{{{k}}}$" for k in ticks])
    ax.set_yticks([]); ax.set_ylim(-0.4,(n-1)*SP+H+0.4)
    for sp in ["left","right","top"]: ax.spines[sp].set_visible(False)
    ax.set_xlabel(f"Exposure dose ({DUNIT[source]}) · log scale")
    ax.set_title(f"{source} — per-farm exposure dose (presumptive MRSA, 10,000-run MC)",
                 fontsize=12.5,fontweight="bold",color=c)
    ax.legend(handles=[Line2D([0],[0],color=c,lw=6,alpha=0.55,label="conditional (given present)"),
                       Line2D([0],[0],color="#333",lw=1.4,ls="--",label="marginal (× occurrence)"),
                       Line2D([0],[0],color="#111",lw=2,label="conditional median")],
              fontsize=8.5,loc="upper right",framealpha=0.9)
    fig.tight_layout(); fig.savefig(fname,dpi=170); plt.close(fig); return fname

# ============================================================================
#  OFAT sensitivity of exposure dose (per source)
# ============================================================================
def sensitivity(fname="exposure_dose_sensitivity.png"):
    S=summary
    specs={
     "Milk":[("C_milk",S.loc["Milk","gm_lo"],S.loc["Milk","grand_mean"],S.loc["Milk","gm_hi"]),
             ("h (cm)",0.0038,0.0046,0.0078),("A (cm2)",3.56,68.7,133.75),
             ("TE_hand-mouth",0.207,0.339,0.471),("f_swallow",0.5,0.75,1.0)],
     "Water":[("C_water",S.loc["Water","gm_lo"],S.loc["Water","grand_mean"],S.loc["Water","gm_hi"]),
              ("V_water (mL)",0.5,1.0,5.0)],
     "Bioaerosol":[("C_air",S.loc["Bioaerosol","gm_lo"],S.loc["Bioaerosol","grand_mean"],S.loc["Bioaerosol","gm_hi"]),
              ("IR (m3/h)",1.5,1.6,1.7),("t (h/day)",4.32,5.58,6.84)],
     "Dust":[("C_dust",S.loc["Dust","gm_lo"],S.loc["Dust","grand_mean"],S.loc["Dust","gm_hi"]),
             ("A (cm2)",3.56,68.7,133.75),("TE_surf-hand",0.01,0.208,0.406),
             ("TE_hand-mouth",0.03,0.10,0.15),("f_swallow",0.5,0.75,1.0),("HtM rate/h",2,2.9,8)],
    }
    fig,axs=plt.subplots(2,2,figsize=(14,8)); axs=axs.ravel()
    for ax,m in zip(axs,SOURCES):
        rows=specs[m]; names=[r[0] for r in rows]
        lo=[100*(r[1]/r[2]-1) for r in rows]; hi=[100*(r[3]/r[2]-1) for r in rows]
        idx=np.argsort([max(abs(a),abs(b)) for a,b in zip(lo,hi)])
        names=[names[i] for i in idx]; lo=[lo[i] for i in idx]; hi=[hi[i] for i in idx]
        y=np.arange(len(names)); c=MC[m]
        for yi,(l,h) in enumerate(zip(lo,hi)):
            ax.barh(yi,h,color=c,alpha=0.85); ax.barh(yi,l,color=c,alpha=0.4)
        ax.axvline(0,color="k",lw=1); ax.set_yticks(y); ax.set_yticklabels(names,fontsize=8.5)
        ax.set_title(f"{m} ({DUNIT[m]})",fontweight="bold",color=c,fontsize=11); ax.set_xlabel("% change in dose")
        ax.grid(axis="x",alpha=0.25)
    fig.suptitle("OFAT sensitivity of exposure dose (dose linear in each factor; concentration from BHM posterior)",
                 fontsize=13,fontweight="bold")
    fig.tight_layout(rect=[0,0,1,0.95]); fig.savefig(fname,dpi=175); plt.close(fig); return fname

# ============================================================================
#  VARIANCE DECOMPOSITION (log scale)  — answers "which input drives the
#  total dose uncertainty?"  Because log D = sum of independent log-terms,
#  Var(log D) = sum_i Var(log factor_i); contribution_i = Var_i / Var(total).
#  This is the proper complement to OFAT (which is only local/conditional).
# ============================================================================
def _logvar_components(source, n=200000):
    r = np.random.default_rng(123)
    def tri(a,c,b): return r.triangular(a,c,b,n)
    comp = {}
    if source == "Milk":
        C = conc_summary("Milk", n)
        comp["Concentration"] = np.log10(C)
        comp["Film h"]        = np.log10(tri(0.0038,0.0046,0.0078))
        comp["Contact area A"]= np.log10(tri(3.56,68.7,133.75))
        comp["TE hand-mouth"] = np.log10(r.beta(4.02,7.85,n))
        comp["f_swallow"]     = np.log10(r.uniform(0.5,1,n))
    elif source == "Water":
        comp["Concentration"] = np.log10(conc_summary("Water", n))
        comp["V_water"]       = np.log10(tri(0.5,1.0,5.0))
    elif source == "Bioaerosol":
        comp["Concentration"] = np.log10(conc_summary("Bioaerosol", n))
        comp["Inhalation IR"] = np.log10(tri(1.5,1.6,1.7))
        comp["Working time t"]= np.log10(tri(4.32,5.58,6.84))
    elif source == "Dust":
        comp["Concentration"] = np.log10(conc_summary("Dust", n))
        comp["Contact area A"]= np.log10(tri(3.56,68.7,133.75))
        comp["TE surf-hand"]  = np.log10(tri(0.01,0.208,0.406))
        comp["TE hand-mouth"] = np.log10(tri(0.03,0.10,0.15))
        comp["f_swallow"]     = np.log10(r.uniform(0.5,1,n))
        comp["HtM rate"]      = np.log10(tri(2,2.9,8))
        comp["Working time t"]= np.log10(tri(4.32,5.58,6.84))
    variances = {k: float(np.var(v)) for k,v in comp.items()}
    tot = sum(variances.values())
    return {k: 100*v/tot for k,v in variances.items()}, tot

def variance_decomposition(fname_bar="dose_variance_decomposition.png",
                           fname_csv="dose_variance_decomposition.csv"):
    rows=[]; contribs={}
    print("\n====  Variance decomposition of log-dose (% of total uncertainty)  ====")
    for m in SOURCES:
        pct, tot = _logvar_components(m); contribs[m]=pct
        conc_share = pct.get("Concentration",0)
        print(f"\n{m}:  (concentration = {conc_share:.0f}% of total log-variance)")
        for k,v in sorted(pct.items(), key=lambda x:-x[1]):
            print(f"    {k:16} {v:5.1f}%"); rows.append(dict(source=m,factor=k,pct_variance=round(v,2)))
    pd.DataFrame(rows).to_csv(fname_csv,index=False)

    # stacked horizontal bar per source (concentration highlighted)
    fig,ax=plt.subplots(figsize=(11,5.2))
    palette=["#C44E52","#4A72A6","#2F8F8B","#E07A5F","#7B5EA7","#8C8C8C","#DDB25E","#5AA469"]
    ypos=np.arange(len(SOURCES))[::-1]
    for yi,m in zip(ypos,SOURCES):
        items=sorted(contribs[m].items(), key=lambda x:-x[1]); left=0
        for j,(k,v) in enumerate(items):
            col="#C44E52" if k=="Concentration" else palette[(j%7)+1]
            ax.barh(yi,v,left=left,color=col,edgecolor="white")
            if v>=6: ax.text(left+v/2,yi,f"{k}\n{v:.0f}%",ha="center",va="center",fontsize=7.5,color="white",fontweight="bold")
            left+=v
    ax.set_yticks(ypos); ax.set_yticklabels(SOURCES,fontweight="bold")
    ax.set_xlabel("% of total log-dose variance"); ax.set_xlim(0,100)
    ax.set_title("Variance decomposition of exposure dose — concentration (red) vs exposure factors\n"
                 "share of total uncertainty (log scale); complements OFAT",fontweight="bold",fontsize=12)
    for sp in ["top","right"]: ax.spines[sp].set_visible(False)
    fig.tight_layout(); fig.savefig(fname_bar,dpi=175); plt.close(fig)

    # pie per source (2x2)
    figp,axs=plt.subplots(2,2,figsize=(12,9)); axs=axs.ravel()
    for ax,m in zip(axs,SOURCES):
        items=sorted(contribs[m].items(), key=lambda x:-x[1])
        labels=[k for k,_ in items]; vals=[v for _,v in items]
        cols=["#C44E52" if k=="Concentration" else palette[(j%7)+1] for j,(k,_) in enumerate(items)]
        ax.pie(vals,labels=labels,autopct="%1.0f%%",colors=cols,textprops={"fontsize":8},
               wedgeprops={"edgecolor":"white"})
        ax.set_title(f"{m} ({DUNIT[m]})",fontweight="bold",color=MC[m])
    figp.suptitle("Variance decomposition per source — concentration (red) vs exposure factors",
                  fontweight="bold",fontsize=13)
    figp.tight_layout(rect=[0,0,1,0.95]); figp.savefig("dose_variance_pie.png",dpi=170); plt.close(figp)
    print(f"\nSaved: {fname_bar}, dose_variance_pie.png, {fname_csv}")
    return contribs

if __name__ == "__main__":
    dose_table()
    for s in SOURCES:
        ridgeline(s)
    sensitivity()
    variance_decomposition()
    print("saved ridgelines: ridge_milk/water/bioaerosol/dust.png  +  exposure_dose_sensitivity.png")

Source     dose unit  occ   conditional  median [95% CrI]     marginal mean
Milk       CFU/event  0.48  6.39 [0.168, 229.5]               16.2
Water      CFU/event  0.85  1.85 [0.595, 5.14]                1.811
Bioaerosol CFU/day    0.89  646 [318, 1306]                   609.8
Dust       CFU/day    0.76  156 [21.6, 721.1]                 161.1
Milk/water = CFU/event ; bioaerosol/dust = CFU/day. Presumptive MRSA. Saved: exposure_dose_summary.csv

====  Variance decomposition of log-dose (% of total uncertainty)  ====

Milk:  (concentration = 86% of total log-variance)
    Concentration     85.6%
    Contact area A     6.8%
    TE hand-mouth      5.7%
    f_swallow          1.2%
    Film h             0.7%

Water:  (concentration = 26% of total log-variance)
    V_water           74.3%
    Concentration     25.7%

Bioaerosol:  (concentration = 93% of total log-variance)
    Concentration     92.8%
    Working time t     6.7%
    Inhalation IR      0.5%

Dust:  (concentration = 15% of to

# Water exposure-dose by type (W1/W2/W3): ridgeline + OFAT + variance decomposition
Water model is 2-level (occurrence=1). Dose/event = (C/100)*V_water. Needs bhm_by_farm_water.csv.
Per-type concentration spread uses detected-replicate log10 (W1 sd 0.48, W2 0.57, W3 0.43).

In [1]:
!pip -q install pandas numpy matplotlib scipy

In [2]:
import os
if not os.path.exists('bhm_by_farm_water.csv'):
    try:
        from google.colab import files; files.upload()
    except Exception as e:
        print('Upload bhm_by_farm_water.csv',e)

Saving bhm_by_farm_water.csv to bhm_by_farm_water.csv


In [3]:
# ============================================================================
#  Water exposure-dose analysis by type (W1 drinking / W2 pre-san / W3 post-san)
#  Water model: 2-level, occurrence = 1 (all detected). Exposure factor = V_water
#  (incidental ingestion). Dose per event:  D = (C_water/100) * V_water  [CFU/event]
#
#  Produces per water type:
#    (1) per-farm ridgeline (conditional filled + marginal dashed; occ=1 so they overlap)
#    (2) OFAT sensitivity (C_water, V_water)
#    (3) variance decomposition (Concentration vs V_water)
#
#  COLAB: !pip -q install pandas numpy matplotlib scipy
#         upload bhm_by_farm_water.csv (from water_refit)  ->  Run all.
# ============================================================================
import numpy as np, pandas as pd, matplotlib
# matplotlib.use("Agg")  # inline on Colab        # remove on Colab for inline
import matplotlib.pyplot as plt
from matplotlib.lines import Line2D
from scipy.stats import gaussian_kde

rng=np.random.default_rng(42); N=10000
byfarm=pd.read_csv("bhm_by_farm_water.csv")
TYPES=["W1","W2","W3"]; LABEL={"W1":"drinking","W2":"pre-sanitation","W3":"post-sanitation"}
COL={"W1":"#7FC9C0","W2":"#2F8F8B","W3":"#1F6B67"}
# per-type overall concentration (log10 mean, sd) from detected replicates (CFU/100 mL)
CTYPE={"W1":(1.614,0.483),"W2":(2.179,0.566),"W3":(1.902,0.426)}

def f_water(n=N): return rng.triangular(0.5,1.0,5.0,n)          # V_water mL (0.5-5.0)
def conc_farm(row,n=N):
    med,lo,hi=row["cond"],row["cond_lo"],row["cond_hi"]
    mu=np.log10(med); s=(np.log10(hi)-np.log10(lo))/(2*1.959964); return 10.0**rng.normal(mu,s,n)
def farm_dose(row,n=N): return (conc_farm(row,n)/100.0)*f_water(n)   # CFU/event

# ---------- (1) per-farm RIDGELINE per water type ----------
def ridgeline(wt,fname):
    d=byfarm[byfarm.matrix==f"Water-{wt}"].sort_values("farm").reset_index(drop=True)
    farms=d["farm"].tolist(); c=COL[wt]
    Dc={}; allpos=[]
    for _,row in d.iterrows():
        cd=farm_dose(row); Dc[row["farm"]]=cd; allpos.append(np.log10(cd[cd>0]))
    allv=np.concatenate(allpos); lo,hi=np.percentile(allv,[0.5,99.5]); xs=np.linspace(lo-0.5,hi+0.5,400)
    n=len(farms); SP=2.4; H=1.35
    fig,ax=plt.subplots(figsize=(9.2,0.95*n+1.8))
    for i,f in enumerate(farms[::-1]):
        y0=i*SP; cd=Dc[f]; occ=1.0     # water occurrence = 1
        lv=np.log10(np.clip(cd[cd>0],1e-8,None)); dens=gaussian_kde(lv)(xs); dens=dens/dens.max()*H
        ax.fill_between(xs,y0,y0+dens,color=c,alpha=0.55,lw=0,zorder=n-i)
        ax.plot(xs,y0+dens,color=c,lw=1.3,zorder=n-i)
        ax.plot(xs,y0+dens*occ,color="#333",lw=1.0,ls="--",zorder=n-i+0.3)   # marginal (=conditional, occ=1)
        ax.plot([np.log10(np.median(cd))]*2,[y0,y0+H*0.9],color="#111",lw=2.0,zorder=n-i+0.5)
        ax.text(xs[-1]+0.05,y0+0.05,"occ 1.00",fontsize=7.5,color="#666",va="bottom")
        ax.text(xs[0]-0.15,y0+0.05,f,ha="right",va="bottom",fontsize=10,fontweight="bold")
    ticks=list(range(int(np.floor(xs[0])),int(np.ceil(xs[-1]))+1))
    ax.set_xticks(ticks); ax.set_xticklabels([f"$10^{{{k}}}$" for k in ticks])
    ax.set_yticks([]); ax.set_ylim(-0.4,(n-1)*SP+H+0.4)
    for sp in ["left","right","top"]: ax.spines[sp].set_visible(False)
    ax.set_xlabel("Exposure dose (CFU/event) \u00b7 log scale")
    ax.set_title(f"Water {wt} ({LABEL[wt]}) \u2014 per-farm exposure dose (10,000-run MC)",
                 fontsize=12.5,fontweight="bold",color=c)
    ax.legend(handles=[Line2D([0],[0],color=c,lw=6,alpha=0.55,label="conditional (given present)"),
                       Line2D([0],[0],color="#333",lw=1.2,ls="--",label="marginal (occ = 1, overlaps)"),
                       Line2D([0],[0],color="#111",lw=2,label="conditional median")],
              fontsize=8.5,loc="upper right",framealpha=0.9)
    fig.tight_layout(); fig.savefig(fname,dpi=170); plt.close(fig); return fname

# ---------- (2) OFAT sensitivity per water type ----------
def sensitivity(wt=None):
    # one SEPARATE figure per water type (concentration uses total per-type spread; V_water 0.5/1/5)
    def one(wt,fname):
        mu,sd=CTYPE[wt]; Cc=10**mu; Clo=10**(mu-1.96*sd); Chi=10**(mu+1.96*sd)
        rows=[("Water concentration C",Clo,Cc,Chi),("Ingestion volume V (mL)",0.5,1.0,5.0)]
        names=[r[0] for r in rows]; lo=[100*(r[1]/r[2]-1) for r in rows]; hi=[100*(r[3]/r[2]-1) for r in rows]
        y=np.arange(len(names)); fig,ax=plt.subplots(figsize=(8,3.2))
        for yi,(l,h) in enumerate(zip(lo,hi)):
            ax.barh(yi,h,color=COL[wt],alpha=0.85); ax.barh(yi,l,color=COL[wt],alpha=0.4)
        ax.axvline(0,color="k",lw=1); ax.set_yticks(y); ax.set_yticklabels(names,fontsize=10)
        ax.set_xlabel("% change in dose"); ax.grid(axis="x",alpha=0.25)
        ax.set_title(f"Water {wt} ({LABEL[wt]}) \u2014 OFAT sensitivity\n(dose linear in each factor)",
                     color=COL[wt],fontweight="bold",fontsize=11)
        fig.tight_layout(); fig.savefig(fname,dpi=175); plt.close(fig); return fname
    if wt: return one(wt,f"water_sens_{wt.lower()}.png")
    return [one(w,f"water_sens_{w.lower()}.png") for w in TYPES]

# ---------- (3) variance decomposition per water type ----------
def variance_decomposition(fname="water_dose_variance.png",fcsv="water_dose_variance.csv"):
    rows=[]; fig,axs=plt.subplots(1,3,figsize=(15,4.2)); axs=axs.ravel()
    for ax,wt in zip(axs,TYPES):
        mu,sd=CTYPE[wt]; r=np.random.default_rng(1); n=200000
        vC=np.var(r.normal(mu,sd,n))                     # log10 concentration variance
        vV=np.var(np.log10(r.triangular(0.5,1.0,5.0,n))) # log10 V_water variance
        tot=vC+vV; pct={"Concentration":100*vC/tot,"V_water":100*vV/tot}
        for k,v in pct.items(): rows.append(dict(type=f"{wt} {LABEL[wt]}",factor=k,pct_variance=round(v,1)))
        cols=["#C44E52","#4A72A6"]
        ax.pie([pct["Concentration"],pct["V_water"]],labels=["Concentration","V_water"],
               autopct="%1.0f%%",colors=cols,textprops={"fontsize":9},wedgeprops={"edgecolor":"white"})
        ax.set_title(f"{wt} ({LABEL[wt]})",color=COL[wt],fontweight="bold")
    pd.DataFrame(rows).to_csv(fcsv,index=False)
    fig.suptitle("Water dose variance decomposition by type \u2014 Concentration (red) vs V_water",
                 fontsize=13,fontweight="bold")
    fig.tight_layout(rect=[0,0,1,0.93]); fig.savefig(fname,dpi=170); plt.close(fig)
    print("variance decomposition:"); print(pd.DataFrame(rows).to_string(index=False)); return fname

if __name__=="__main__":
    for wt in TYPES: ridgeline(wt,f"water_ridge_{wt.lower()}.png")
    sensitivity(); variance_decomposition()
    print("saved: water_ridge_w1/w2/w3.png, water_sens_w1/w2/w3.png, water_dose_variance.png (+csv)")


variance decomposition:
              type        factor  pct_variance
       W1 drinking Concentration          83.5
       W1 drinking       V_water          16.5
 W2 pre-sanitation Concentration          87.4
 W2 pre-sanitation       V_water          12.6
W3 post-sanitation Concentration          79.8
W3 post-sanitation       V_water          20.2
saved: water_ridge_w1/w2/w3.png, water_sens_w1/w2/w3.png, water_dose_variance.png (+csv)
